# iGaming Player Churn Prediction
## 01 - Raw data validation

**Objective:** Validate the structure, completeness, player identifiers, dates, and numeric values of the local bwin Virtual Casino Gambling source data before feature engineering or machine learning.

Source text files are read directly from ZIP archives. All raw files and source records are preserved, including duplicate and unusual records. This notebook creates no churn labels, predictive features, models, or processed exports.

**Inputs:** the demographics ZIP, daily activity ZIP, and accompanying PDF codebook in `data/raw/`. Raw data is intentionally excluded from Git. Reproducing these results requires the same local source files and the repository's Python environment.

A successful execution confirms readability and completes the checks; it does not imply all records are ready for downstream analysis.

### 1. Environment and project paths

Locate the repository from the working directory and its parents. This supports execution from either the repository root or `notebooks/`, without machine-specific absolute paths. `PROCESSED_DIR` is defined for later work, but is not written to.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 60)
pd.set_option('display.precision', 3)

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'data/raw').is_dir()
     and (p / 'notebooks/01_data_validation.ipynb').is_file()), None
)
if project_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
RAW_DIR = project_root / 'data' / 'raw'
PROCESSED_DIR = project_root / 'data' / 'processed'
print('Project root located; source directory: data/raw/')
print(f'pandas {pd.__version__}; numpy {np.__version__}')

Project root located; source directory: data/raw/
pandas 3.0.6; numpy 2.5.3


### 2. Verify files and inspect archives

Check all three expected files, print all ZIP member names, and test archive integrity before loading. Require exactly one text file per archive rather than silently choosing an ambiguous member. Nothing is extracted.

In [2]:
expected_sources = [
    'Codebook_for_Virtual_Casino_Gambling.pdf',
    'RawDataSet1_DemographicsCasinoTXT.zip',
    'RawDataSet2_DailyAggregCasinoTXT.zip',
]
source_inventory = pd.DataFrame([
    {'file': name, 'exists': (RAW_DIR / name).is_file(),
     'bytes': (RAW_DIR / name).stat().st_size if (RAW_DIR / name).is_file() else np.nan}
    for name in expected_sources
])
display(source_inventory)
missing_sources = source_inventory.loc[~source_inventory['exists'], 'file'].tolist()
if missing_sources:
    raise FileNotFoundError(f'Missing source files: {missing_sources}')

archive_members = {}
for name in expected_sources[1:]:
    with zipfile.ZipFile(RAW_DIR / name) as archive:
        print(f'\n{name}')
        for member in archive.infolist():
            print(f'  {member.filename} ({member.file_size:,} uncompressed bytes)')
        bad_member = archive.testzip()
        if bad_member is not None:
            raise ValueError(f'Archive integrity failure: {bad_member}')
        text_members = [m.filename for m in archive.infolist()
                        if not m.is_dir() and Path(m.filename).suffix.lower() == '.txt']
        if len(text_members) != 1:
            raise ValueError(f'Expected exactly one text member in {name}: {text_members}')
        archive_members[name] = text_members[0]

,file,exists,bytes
0,Codebook_for_Virtual_Casino_Gambling.pdf,True,114398
1,RawDataSet1_DemographicsCasinoTXT.zip,True,33054
2,RawDataSet2_DailyAggregCasinoTXT.zip,True,1023224



RawDataSet1_DemographicsCasinoTXT.zip
  RawDataSet1_DemographicsCasinoTXT.txt (195,138 uncompressed bytes)

RawDataSet2_DailyAggregCasinoTXT.zip
  RawDataSet2_DailyAggregCasinoTXT.txt (3,165,476 uncompressed bytes)


**Observed:** All three inputs exist. Both ZIPs pass integrity checks and contain one text file each: `RawDataSet1_DemographicsCasinoTXT.txt` and `RawDataSet2_DailyAggregCasinoTXT.txt`. The codebook PDF is verified for presence; category mappings and unusual-record semantics still require documentation review.

### 3. Load and inspect the tables

Headers show tab-separated fields. Use an explicit tab separator and UTF-8 encoding. Preserve `UserID` as a nullable integer identifier, then validate the schema and display shapes, first rows, column names, and loaded dtypes. Category codes remain unlabelled until their mappings are verified.

In [3]:
def read_source(archive_name):
    with zipfile.ZipFile(RAW_DIR / archive_name) as archive:
        with archive.open(archive_members[archive_name]) as source:
            return pd.read_csv(source, sep='\t', encoding='utf-8', dtype={'UserID': 'Int64'})

demographics = read_source(expected_sources[1])
daily_activity = read_source(expected_sources[2])
expected_columns = {
    'Demographics': ['UserID', 'Country', 'Language', 'Gender', 'RegDate', 'Fstcadate', 'Fstpdate'],
    'Daily activity': ['UserID', 'Date', 'Stake', 'Winnings', 'Bets'],
}
for name, frame in [('Demographics', demographics), ('Daily activity', daily_activity)]:
    if list(frame.columns) != expected_columns[name]:
        raise ValueError(f'Unexpected {name} schema: {list(frame.columns)}')
    print(f'\n{name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns')
    print('Columns:', frame.columns.tolist())
    display(frame.head(5))
    display(frame.dtypes.rename('dtype').to_frame())


Demographics: 4,222 rows x 7 columns
Columns: ['UserID', 'Country', 'Language', 'Gender', 'RegDate', 'Fstcadate', 'Fstpdate']


,UserID,Country,Language,Gender,RegDate,Fstcadate,Fstpdate
0,1324364,276,2,1,2/1/2005,8/26/2006,2/3/2005
1,1324369,792,7,1,2/1/2005,3/1/2005,2/1/2005
2,1324377,208,11,1,2/1/2005,4/4/2005,2/1/2005
3,1324383,578,10,1,2/1/2005,6/5/2005,2/3/2005
4,1324386,756,1,1,2/1/2005,2/10/2005,2/9/2005


,dtype
UserID,Int64
Country,int64
Language,int64
Gender,int64
RegDate,str
Fstcadate,str
Fstpdate,str



Daily activity: 102,165 rows x 5 columns
Columns: ['UserID', 'Date', 'Stake', 'Winnings', 'Bets']


,UserID,Date,Stake,Winnings,Bets
0,1363150,7/5/2006,485.0,-150.0,4
1,1383941,8/21/2006,16.5,-121.0,8
2,1391558,9/8/2006,4.0,-100.0,2
3,1333544,2/25/2007,0.0,-91.0,0
4,1345514,12/28/2006,0.0,-50.0,0


,dtype
UserID,Int64
Date,str
Stake,float64
Winnings,float64
Bets,int64


**Observed:** Demographics contains **4,222 rows x 7 columns**; daily activity contains **102,165 rows x 5 columns**. Schemas match the expected fields. Dates load as text; stake and winnings are floating-point values and bets are integers. Date parsing follows next.

### 4. Parse and audit all dates

The source uses **month/day/year**, visible in values such as `8/26/2006`. Parse with `%m/%d/%Y`, report existing missing values separately from parsing failures, and preserve original date strings in memory for audit. Invalid nonmissing dates cause an explicit failure after examples are displayed.

In [4]:
raw_date_values = {}
date_audit_rows = []
for name, frame, columns in [
    ('Demographics', demographics, ['RegDate', 'Fstcadate', 'Fstpdate']),
    ('Daily activity', daily_activity, ['Date']),
]:
    for column in columns:
        original = frame[column].copy()
        raw_date_values[(name, column)] = original
        parsed = pd.to_datetime(original, format='%m/%d/%Y', errors='coerce')
        invalid = original.notna() & parsed.isna()
        date_audit_rows.append({
            'dataset': name, 'column': column,
            'source_missing': int(original.isna().sum()),
            'parse_failures': int(invalid.sum()),
            'start': parsed.min(), 'end': parsed.max(),
        })
        if invalid.any():
            display(original.loc[invalid].head(10).to_frame('invalid_source_date'))
        frame[column] = parsed

date_audit = pd.DataFrame(date_audit_rows)
display(date_audit)
if date_audit['parse_failures'].sum():
    raise ValueError('Date parsing failures require review; no rows were deleted.')
for name, frame in [('Demographics', demographics), ('Daily activity', daily_activity)]:
    print(name)
    display(frame.dtypes.rename('parsed_dtype').to_frame())

,dataset,column,source_missing,parse_failures,start,end
0,Demographics,RegDate,0,0,2005-02-01,2005-02-28
1,Demographics,Fstcadate,0,0,2005-02-01,2007-01-30
2,Demographics,Fstpdate,0,0,2005-02-01,2007-01-25
3,Daily activity,Date,0,0,2005-02-01,2007-02-28


Demographics


,parsed_dtype
UserID,Int64
Country,int64
Language,int64
Gender,int64
RegDate,datetime64[us]
Fstcadate,datetime64[us]
Fstpdate,datetime64[us]


Daily activity


,parsed_dtype
UserID,Int64
Date,datetime64[us]
Stake,float64
Winnings,float64
Bets,int64


**Observed:** All four date columns parse successfully with **0 missing dates** and **0 parsing failures**.

| Date field | Earliest | Latest |
| --- | --- | --- |
| Registration (`RegDate`) | 2005-02-01 | 2005-02-28 |
| First casino activity (`Fstcadate`) | 2005-02-01 | 2007-01-30 |
| First pay-in (`Fstpdate`) | 2005-02-01 | 2007-01-25 |
| Daily activity (`Date`) | 2005-02-01 | 2007-02-28 |

These are observed source ranges, not an assumed churn or observation window.

### 5. Demographic completeness and identifier uniqueness

Distinguish all rows involved in duplicate IDs from extra rows beyond the first per ID. Report missingness for every column after date parsing. No rows are removed.

In [5]:
def missing_table(frame):
    return pd.DataFrame({
        'missing_count': frame.isna().sum(),
        'missing_percent': frame.isna().mean().mul(100),
    })

demographic_duplicate_mask = demographics.duplicated('UserID', keep=False)
demographic_summary = pd.Series({
    'total_rows': len(demographics),
    'unique_UserIDs': demographics['UserID'].nunique(),
    'rows_with_duplicate_UserID': int(demographic_duplicate_mask.sum()),
    'extra_rows_beyond_first_UserID': int(demographics.duplicated('UserID').sum()),
}, name='count')
display(demographic_summary.to_frame())
display(missing_table(demographics))
if demographic_duplicate_mask.any():
    display(demographics.loc[demographic_duplicate_mask].sort_values('UserID').head(10))

,count
total_rows,4222
unique_UserIDs,4222
rows_with_duplicate_UserID,0
extra_rows_beyond_first_UserID,0


,missing_count,missing_percent
UserID,0,0.0
Country,0,0.0
Language,0,0.0
Gender,0,0.0
RegDate,0,0.0
Fstcadate,0,0.0
Fstpdate,0,0.0


**Observed:** The demographic table contains **4,222 unique players** across **4,222 rows**, with **0 duplicate UserIDs** and **0 missing values in every column**. Its observed grain is one record per player.

### 6. Country, Language, and Gender distributions

Display every observed code with a count and percentage. Include missing categories if present, and avoid assigning unverified category labels.

In [6]:
category_distributions = {}
for column in ['Country', 'Language', 'Gender']:
    counts = demographics[column].value_counts(dropna=False)
    distribution = counts.rename('players').to_frame()
    distribution['percent'] = counts.div(len(demographics)).mul(100)
    distribution.index.name = column
    category_distributions[column] = distribution
    print(f'\n{column}: {len(distribution)} observed categories')
    display(distribution)


Country: 46 observed categories


,players,percent
Country,,
276,784,18.569
300,482,11.416
40,468,11.085
724,425,10.066
250,373,8.835
208,355,8.408
380,318,7.532
792,317,7.508
616,223,5.282



Language: 19 observed categories


,players,percent
Language,,
2,1419,33.610
8,488,11.559
6,398,9.427
4,395,9.356
11,347,8.219
7,318,7.532
3,313,7.414
9,216,5.116
1,169,4.003



Gender: 2 observed categories


,players,percent
Gender,,
1,3924,92.942
0,298,7.058


**Observed:** There are **46 Country codes**, **19 Language codes**, and **2 Gender codes**. Country code `276` is most frequent (784 players), as is Language code `2` (1,419 players). Gender codes `1` and `0` occur 3,924 and 298 times respectively. These counts describe this cohort; no representativeness or category meanings are inferred.

### 7. Daily completeness and numeric validation

Report row and player counts, date range, missingness, and descriptive statistics. Count negative, zero, and nonfinite values without changing them. Display examples for negative winnings, zero stake, and zero bets to support review.

In [7]:
metrics = ['Stake', 'Winnings', 'Bets']
for column in metrics:
    if not pd.api.types.is_numeric_dtype(daily_activity[column]):
        raise TypeError(f'{column} must be numeric.')
daily_summary = pd.Series({
    'total_rows': len(daily_activity),
    'unique_UserIDs': daily_activity['UserID'].nunique(),
    'date_start': daily_activity['Date'].min(),
    'date_end': daily_activity['Date'].max(),
}, name='value')
display(daily_summary.to_frame())
display(missing_table(daily_activity))
display(daily_activity[metrics].describe().T)
numeric_checks = pd.DataFrame({
    'negative_count': daily_activity[metrics].lt(0).sum(),
    'zero_count': daily_activity[metrics].eq(0).sum(),
    'nonfinite_count': (~np.isfinite(daily_activity[metrics])).sum(),
})
display(numeric_checks)
for label, mask in [
    ('Negative winnings', daily_activity['Winnings'].lt(0)),
    ('Zero stake', daily_activity['Stake'].eq(0)),
    ('Zero bets', daily_activity['Bets'].eq(0)),
]:
    print(f'\n{label}: examples')
    display(daily_activity.loc[mask].head(5))

,value
total_rows,102165
unique_UserIDs,4222
date_start,2005-02-01 00:00:00
date_end,2007-02-28 00:00:00


,missing_count,missing_percent
UserID,0,0.0
Date,0,0.0
Stake,0,0.0
Winnings,0,0.0
Bets,0,0.0


,count,mean,std,min,25%,50%,75%,max
Stake,102165.0,1122.874,4648.325,0.0,30.00,155.0,682.50,308002.1
Winnings,102165.0,1088.173,4644.713,-150.0,22.25,136.0,637.75,344539.5
Bets,102165.0,145.260,369.038,0.0,4.00,18.0,112.00,10817.0


,negative_count,zero_count,nonfinite_count
Stake,0,5044,0
Winnings,72,9309,0
Bets,0,49,0



Negative winnings: examples


,UserID,Date,Stake,Winnings,Bets
0,1363150,2006-07-05,485.0,-150.0,4
1,1383941,2006-08-21,16.5,-121.0,8
2,1391558,2006-09-08,4.0,-100.0,2
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0



Zero stake: examples


,UserID,Date,Stake,Winnings,Bets
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0
8,1335498,2005-03-05,0.0,-25.0,0
14,1350206,2006-08-01,0.0,-15.0,0
16,1364109,2005-03-02,0.0,-13.0,0



Zero bets: examples


,UserID,Date,Stake,Winnings,Bets
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0
8,1335498,2005-03-05,0.0,-25.0,0
14,1350206,2006-08-01,0.0,-15.0,0
16,1364109,2005-03-02,0.0,-13.0,0


**Observed:** There are **102,165 records for 4,222 players** and **0 missing or nonfinite numeric values**. Stake has **0 negatives** and **5,044 zeros**. Winnings has **72 negatives** (minimum **-150**) and **9,309 zeros**. Bets has **0 negatives** and **49 zeros**.

Stake and winnings means are about 1,122.87 and 1,088.17, versus medians of 155 and 136. Their maxima are 308,002.10 and 344,539.50. These differences warrant further distribution review. The cause of negative winnings and zero stake/bet records is not established here; review the source documentation before making corrections or exclusions.

### 8. Duplicate player-date combinations

Count **all rows belonging to repeated keys**, **distinct repeated keys**, and **extra rows beyond the first per key** separately. Fully identical duplicate rows are another distinct check. Display examples and multiplicities; preserve all records without deduplication or aggregation.

In [8]:
player_date_key = ['UserID', 'Date']
duplicate_mask = daily_activity.duplicated(player_date_key, keep=False)
duplicate_records = daily_activity.loc[duplicate_mask].sort_values(player_date_key)
repeated_key_sizes = duplicate_records.groupby(player_date_key, dropna=False).size()
duplicate_summary = pd.Series({
    'rows_in_repeated_player_date_keys': int(duplicate_mask.sum()),
    'distinct_repeated_player_date_keys': len(repeated_key_sizes),
    'extra_rows_beyond_first_per_player_date': int(daily_activity.duplicated(player_date_key).sum()),
    'fully_identical_extra_rows': int(daily_activity.duplicated().sum()),
}, name='count')
display(duplicate_summary.to_frame())
print('Repeated player-date examples:')
display(duplicate_records.head(12))
print('Multiplicity of repeated keys:')
display(repeated_key_sizes.value_counts().sort_index().rename_axis('rows_per_key').to_frame('key_count'))

,count
rows_in_repeated_player_date_keys,6334
distinct_repeated_player_date_keys,3160
extra_rows_beyond_first_per_player_date,3174
fully_identical_extra_rows,1


Repeated player-date examples:


,UserID,Date,Stake,Winnings,Bets
21284,1324369,2005-12-20,23.00,14.50,8
21453,1324369,2005-12-20,25.00,15.00,25
53438,1324369,2006-01-10,197.00,156.50,19
55681,1324369,2006-01-10,179.00,179.50,99
68405,1324369,2006-01-21,423.00,380.00,14
84723,1324369,2006-01-21,1208.00,1158.00,279
20174,1324369,2006-07-05,11.00,12.50,11
50836,1324369,2006-07-05,167.00,134.50,17
10985,1324369,2007-01-06,5.00,2.00,1
34374,1324369,2007-01-06,61.50,46.50,60


Multiplicity of repeated keys:


,key_count
rows_per_key,
2,3146
3,14


**Observed:** **6,334 rows** belong to **3,160 repeated UserID-Date keys**, corresponding to **3,174 extra rows beyond one per key**. Only **1 extra row is fully identical**. Most repeated keys therefore contain different records; discarding them automatically would lose source information. Their meaning and handling policy need review before downstream work.

### 9. Cross-table player coverage

Compare nonmissing identifier sets in both directions. This tests cohort alignment, not the meaning or temporal consistency of each record.

In [9]:
demographic_ids = set(demographics['UserID'].dropna())
activity_ids = set(daily_activity['UserID'].dropna())
ids_in_both = demographic_ids & activity_ids
ids_only_demographics = demographic_ids - activity_ids
ids_only_activity = activity_ids - demographic_ids
id_coverage = pd.Series({
    'players_in_both': len(ids_in_both),
    'players_only_demographics': len(ids_only_demographics),
    'players_only_daily_activity': len(ids_only_activity),
}, name='players')
display(id_coverage.to_frame())
print('Player ID sets match:', demographic_ids == activity_ids)
if ids_only_demographics:
    print('Demographics-only examples:', sorted(ids_only_demographics)[:10])
if ids_only_activity:
    print('Daily-only examples:', sorted(ids_only_activity)[:10])

,players
players_in_both,4222
players_only_demographics,0
players_only_daily_activity,0


Player ID sets match: True


**Observed:** The identifier sets match exactly: **4,222 players appear in both**, with **0 demographics-only** and **0 daily-activity-only players**. No unmatched IDs are present in these source tables.

### 10. Computed validation summary

This final table derives its values from the loaded data. Markdown observations describe the saved run; update those interpretations if the source inputs change.

In [10]:
validation_summary = pd.Series({
    'demographics_shape': str(demographics.shape),
    'daily_activity_shape': str(daily_activity.shape),
    'demographic_players': len(demographic_ids),
    'daily_players': len(activity_ids),
    'demographic_missing_cells': int(demographics.isna().sum().sum()),
    'daily_missing_cells': int(daily_activity.isna().sum().sum()),
    'date_parse_failures': int(date_audit['parse_failures'].sum()),
    'rows_in_repeated_player_date_keys': int(duplicate_mask.sum()),
    'distinct_repeated_player_date_keys': len(repeated_key_sizes),
    'extra_player_date_rows': int(daily_activity.duplicated(player_date_key).sum()),
    'negative_stake_rows': int(numeric_checks.loc['Stake', 'negative_count']),
    'negative_winnings_rows': int(numeric_checks.loc['Winnings', 'negative_count']),
    'negative_bets_rows': int(numeric_checks.loc['Bets', 'negative_count']),
    'zero_stake_rows': int(numeric_checks.loc['Stake', 'zero_count']),
    'zero_bets_rows': int(numeric_checks.loc['Bets', 'zero_count']),
    'player_ID_sets_match': demographic_ids == activity_ids,
}, name='result')
display(validation_summary.to_frame())
display(date_audit[['dataset', 'column', 'start', 'end']])
print('Validation complete. All source records retained; no processed files written.')

,result
demographics_shape,"(4222, 7)"
daily_activity_shape,"(102165, 5)"
demographic_players,4222
daily_players,4222
demographic_missing_cells,0
daily_missing_cells,0
date_parse_failures,0
rows_in_repeated_player_date_keys,6334
distinct_repeated_player_date_keys,3160
extra_player_date_rows,3174


,dataset,column,start,end
0,Demographics,RegDate,2005-02-01,2005-02-28
1,Demographics,Fstcadate,2005-02-01,2007-01-30
2,Demographics,Fstpdate,2005-02-01,2007-01-25
3,Daily activity,Date,2005-02-01,2007-02-28


Validation complete. All source records retained; no processed files written.


### Review before the next stage

- Resolve repeated player-date records against source documentation before choosing any deduplication or aggregation policy.
- Review negative winnings and zero stake/bet records before deciding how to handle them.
- Confirm category mappings and measurement units from the codebook before interpreting codes or amounts.
- Review timing and distributions before defining an observation window or churn outcome.

**Day 1 boundary:** Validation only. No rows have been removed, raw files modified, or processed datasets saved. Feature engineering, churn labelling, and machine learning remain outside the current scope.